In [1]:
import pandas as pd
import numpy as np
import sys
import warnings
import gc
warnings.filterwarnings('ignore')
gc.disable()
import matplotlib.pyplot as plt
import matplotlib.gridspec as gs
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from copy import deepcopy,copy
from ipywidgets import IntProgress
from itertools import chain
from IPython.display import display
from datetime import datetime
import pickle
import os
from sympy import sympify,latex,Float,simplify
import random
from math import ceil,sqrt
import seaborn as sbrn
from scipy.optimize import curve_fit
import pynumdiff
# Catch stout
from io import StringIO 
import sys
from contextlib import redirect_stdout
# Since the 'user' column do not have relevant information will not be read

# Import Machine Scientist
from importlib.machinery import SourceFileLoader
path = '/export/home/oriolca/BMS_ODE/Bacteries/rguimera-machine-scientist-linear-term/machinescientist_ode.py'
ms = SourceFileLoader('ms', path).load_module()

2025-03-24 20:23:14,667 [INFO] 
Limited Total Variation Regularization Support Detected! 
---> CVXPY is not installed. 
---> Many Total Variation Methods require CVXPY including: 
---> velocity, acceleration, jerk, jerk_sliding, smooth_acceleration
---> Please install CVXPY to use these methods.
---> Recommended to also install MOSEK and obtain a MOSEK license.
You can still use: total_variation_regularization.iterative_velocity

2025-03-24 20:23:14,668 [INFO] 
Limited Linear Model Support Detected! 
---> PYCHEBFUN is not installed. 
---> Install pychebfun to use chebfun derivatives (https://github.com/pychebfun/pychebfun/) 
You can still use other methods 

2025-03-24 20:23:14,669 [INFO] 
Limited Linear Model Support Detected! 
---> CVXPY is not installed. 
---> Install CVXPY to use lineardiff derivatives 
You can still use other methods 



In [2]:
folder_name = 'Train_test_data_lin_term_com2025_03_11-11_21_44'
with open(f'./{folder_name}/x.pkl', 'rb') as file:
    # A new file will be created
    x = pickle.load(file)
    
with open(f'./{folder_name}/y.pkl', 'rb') as file:
    # A new file will be created
    y = pickle.load(file)

In [5]:
f_name_train='Full_data_lin_term_com_2025_03_27-06_00_27/'
try:
    file = open(f'./{f_name_train}/model_mdl.pkl','rb')
except:
    file = open(f'./{f_name_train}/mdl.pkl','rb')
bms_fulldata = pickle.load(file) 
print(bms_fulldata.E)

####################
#Generate new model fromstring
str = f'{bms_fulldata.constraint[0]}{bms_fulldata.pr(show_pow=True)}{bms_fulldata.constraint[1]}'
print(str)
bms_fulldata_new = ms.from_string_model(x, y,str , 1, 8, ['B'])
print(bms_fulldata_new.E)
old_par_values=deepcopy(bms_fulldata.par_values)
old_cols=list(old_par_values.keys())

bms_fulldata_new.fit_par={}
bms_fulldata_new.par_values=deepcopy(bms_fulldata.par_values)
bms_fulldata_new.get_bic(reset=True,fit=True)
bms_fulldata_new.get_energy(reset=True)
initial_fit_pars=deepcopy(bms_fulldata_new.par_values)
f_name=f'{f_name_train}mdl_refit_train.pkl'
file = open(f_name,'wb')
pickle.dump(bms_fulldata_new,file)
file.close()
print(bms_fulldata_new.E)
old_energy = deepcopy(bms_fulldata_new.E)

for o_col in old_cols[1:]:
    for n_col in old_cols:
        #print(o_col,n_col)
        #test_model=deepcopy(bms_fulldata_new)
        bms_fulldata_new.fit_par={}
        bms_fulldata_new.par_values[n_col]=deepcopy(old_par_values[o_col])
        bms_fulldata_new.get_bic(reset=True,fit=True)
        bms_fulldata_new.get_energy(reset=True)
        if bms_fulldata_new.E<old_energy:
            print('Better fit. Update model.',bms_fulldata_new.E)
            file = open(f_name,'wb')
            pickle.dump(bms_fulldata_new,file)
            file.close()
            old_energy=deepcopy(bms_fulldata_new.E)
        else:
            # Not improving energy. Setting previorus parameters
            bms_fulldata_new.par_values[n_col]=deepcopy(initial_fit_pars[n_col])

-16729.6565686572
((_a0_ * B) + ((exp((pow2(pow3(B)) * (_a2_ / _a5_))) + _a5_) * _a7_))
Model summary
Par_values: {'B5': {'_a0_': -0.02150226547382772, '_a2_': -934928542209.8938, '_a5_': 6428.161256341401, '_a7_': 4.2199661349773265e-06}, 'A8': {'_a0_': 2034764.310447023, '_a2_': 3422633593927.2, '_a5_': -2034764.239894228, '_a7_': 1.0}, 'G7': {'_a0_': 79815.73819106343, '_a2_': 6526005.767236352, '_a5_': -79815.73493866633, '_a7_': 1.0}, 'A9': {'_a0_': 855.7403134967201, '_a2_': 694739547.8018352, '_a5_': -855.706083921435, '_a7_': 1.0}, 'F7': {'_a0_': -1.0709213510167606, '_a2_': -8.304450892472699e+51, '_a5_': 2.901305910196427e+31, '_a7_': 1.0}, 'G8': {'_a0_': 6009.209559438243, '_a2_': 58091.24678397918, '_a5_': -6010.0619770529665, '_a7_': 1.0}, 'D12': {'_a0_': 0.3958255125375106, '_a2_': 28.060171622901827, '_a5_': -1.1270332830125849, '_a7_': 0.25418892965119927}, 'H12': {'_a0_': -1.5489964136749756, '_a2_': 732.2386341396572, '_a5_': -40.85398067893574, '_a7_': 0.999966345725


KeyboardInterrupt

